# Bronze Layer: Raw API Data Ingestion

**Data Source:** FastAPI REST endpoint (http://127.0.0.1:8000/upload)  
**Destination:** S3 Delta table (s3://sales-pipeline-data-lake-ratnajit/bronze/customer_support_tickets/)

**Approach:**
- API extraction with pagination (100 records per page)
- Error handling for rate limits, server errors, timeouts
- Retry logic with exponential backoff
- Raw data stored as STRING (no transformations)
- Metadata columns added for lineage tracking

In [0]:
import requests
import time
import uuid
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

# API config
API_URL = 'http://127.0.0.1:8000/upload'
PAGE_SIZE = 100
MAX_RETRIES = 3
TIMEOUT = 30

# Bronze table config
BRONZE_TABLE = 'workspace.bronze.customer_support_tickets'
BRONZE_S3_PATH = 's3://sales-pipeline-data-lake-ratnajit/bronze/customer_support_tickets/'

BATCH_ID = str(uuid.uuid4())
print(f'Starting ingestion - Batch ID: {BATCH_ID}')

In [0]:
spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.bronze')

# Define schema - bronze layer stores everything as STRING
schema = StructType([
    StructField('ticket_id', StringType()),
    StructField('agent_name', StringType()),
    StructField('customer_id', StringType()),
    StructField('issue_type', StringType()),
    StructField('status', StringType()),
    StructField('aht_minutes', StringType()),
    StructField('created_date', StringType()),
])

page = 1
total_records = 0

print(f'Starting extraction from {API_URL}')

# Pagination loop - fetch 100 records per page
while True:
    print(f'\nFetching page {page}')
    
    # Retry logic with exponential backoff (1s, 2s, 4s)
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(
                API_URL,
                params={'page': page, 'page_size': PAGE_SIZE},
                timeout=TIMEOUT
            )
            response.raise_for_status()
            data = response.json()
            print(f'  Page {page} fetched successfully')
            break
            
        except requests.exceptions.Timeout:
            if attempt < MAX_RETRIES - 1:
                wait_time = 2 ** attempt
                print(f'  Timeout - retrying in {wait_time}s')
                time.sleep(wait_time)
            else:
                raise
                
        except requests.exceptions.ConnectionError:
            if attempt < MAX_RETRIES - 1:
                wait_time = 2 ** attempt
                print(f'  Connection error - retrying in {wait_time}s')
                time.sleep(wait_time)
            else:
                raise
                
        except requests.exceptions.HTTPError as e:
            # Handle rate limiting and server errors
            status_code = e.response.status_code if e.response else None
            if status_code in [429, 500, 502, 503, 504] and attempt < MAX_RETRIES - 1:
                wait_time = 2 ** attempt
                print(f'  HTTP {status_code} - retrying in {wait_time}s')
                time.sleep(wait_time)
            else:
                raise
    
    records = data if isinstance(data, list) else data.get('data', [])
    if not records:
        print(f'  No more data - stopping')
        break
    
    # Convert to Spark DataFrame
    df = spark.createDataFrame(records, schema=schema)
    
    # Add metadata columns for data lineage
    df_bronze = df.select(
        '*',
        current_timestamp().alias('_ingested_at'),
        lit(f'{API_URL}?page={page}').alias('_source_url'),
        lit(BATCH_ID).alias('_batch_id'),
        lit(page).alias('_page')
    )
    
    # Write to S3 Delta table (append mode for bronze)
    df_bronze.write \
        .format('delta') \
        .mode('append') \
        .option('path', BRONZE_S3_PATH) \
        .saveAsTable(BRONZE_TABLE)
    
    record_count = len(records)
    total_records += record_count
    print(f'  Saved {record_count} records (total: {total_records})')
    
    # Stop if we got fewer records than PAGE_SIZE (last page)
    if record_count < PAGE_SIZE:
        break
    
    page += 1

print(f'\nExtraction complete')
print(f'Total: {total_records} records across {page} pages')
print(f'Batch ID: {BATCH_ID}')

In [0]:
# CSV fallback - we use this when API is unavailable
# In production, Cell 3 handles API extraction

import uuid
from pyspark.sql.functions import lit, current_timestamp

BRONZE_TABLE = 'workspace.bronze.customer_support_tickets'
BRONZE_S3_PATH = 's3://sales-pipeline-data-lake-ratnajit/bronze/customer_support_tickets/'
CSV_PATH = 'idbfs:/2026-09-28/21/_5fcc1a59-e317-4a7f-bf23-6e81f6f8c2e1'

BATCH_ID = str(uuid.uuid4())
print(f'Loading from CSV: {CSV_PATH}')

# Read CSV - keep all columns as STRING
df = spark.read.csv(CSV_PATH, header=True, inferSchema=False).select(
    'ticket_id', 'agent_name', 'customer_id', 'issue_type',
    'status', 'aht_minutes', 'created_date'
)

# Add same metadata columns as API extraction
df_bronze = df.select(
    '*',
    current_timestamp().alias('_ingested_at'),
    lit(CSV_PATH).alias('_source_url'),
    lit(BATCH_ID).alias('_batch_id'),
    lit(1).alias('_page')
)

# Write to S3 and register table
df_bronze.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .save(BRONZE_S3_PATH)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE}
USING DELTA
LOCATION '{BRONZE_S3_PATH}'
""")

print(f'Loaded {df_bronze.count()} records')
print(f'Batch ID: {BATCH_ID}')

Loading from CSV: idbfs:/2026-09-28/21/_5fcc1a59-e317-4a7f-bf23-6e81f6f8c2e1
Loaded 1000 records
Batch ID: 7d85815a-c003-488c-8bc3-a00747bec6d5


In [0]:
# Verify bronze table by reading directly from S3
df = spark.read.format('delta').load(BRONZE_S3_PATH)

print(f'Total records in bronze: {df.count()}')
print(f'Records from current batch: {df.filter(f"_batch_id = \'{BATCH_ID}\'").count()}')

print('\nSample records:')
df.show(5, truncate=False)

print('\nBatch summary:')
df.groupBy('_batch_id').count() \
    .withColumnRenamed('count', 'records') \
    .orderBy('_batch_id') \
    .show(truncate=False)

Total records in bronze: 1000
Records from current batch: 1000

Sample records:
+---------+----------+-----------+----------+--------+-----------+------------+--------------------------+----------------------------------------------------------+------------------------------------+-----+
|ticket_id|agent_name|customer_id|issue_type|status  |aht_minutes|created_date|_ingested_at              |_source_url                                               |_batch_id                           |_page|
+---------+----------+-----------+----------+--------+-----------+------------+--------------------------+----------------------------------------------------------+------------------------------------+-----+
|T001     |Rahul     |C101       |Billing   |Resolved|18         |2026-09-25  |2026-09-29 00:30:34.723647|idbfs:/2026-09-28/21/_5fcc1a59-e317-4a7f-bf23-6e81f6f8c2e1|7d85815a-c003-488c-8bc3-a00747bec6d5|1    |
|T002     |Priya     |C102       |Technical |Resolved|32         |2026-09-25  |2026-